# Тахи · сегментация частей тела (уши / ноги / тело)

DINOv3 ViT-S (замороженный) + линейная голова на патчах. Обучает 4 класса: фон, уши, ноги, тело. После обучения — self-training псевдоразметкой на неразмеченном пуле и экспорт TorchScript `horse_parts.pt` для локального сервиса.

**Перед запуском:**
1. Прикрепите Dataset `takhi-parts` (папка `images/` + `labels.json`).
2. Опционально прикрепите `takhi-unlabeled` для self-training.
3. Добавьте секрет `HF_TOKEN` (Add-ons → Secrets) с токеном Hugging Face, у которого принята лицензия DINOv3.
4. Accelerator: GPU P100 (минимум).

In [ ]:
!pip install -q -U transformers torch torchvision pillow huggingface_hub

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ.setdefault("HF_TOKEN", UserSecretsClient().get_secret("HF_TOKEN"))
except Exception:
    pass
from huggingface_hub import login
login(token=os.environ["HF_TOKEN"])

In [ ]:
import json
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image, ImageDraw, ImageOps
from transformers import AutoImageProcessor, DINOv3ViTModel

CLASSES = ["background", "ears", "legs", "body"]
SIZE, PATCH, PATCHES, OFFSET = 448, 16, 28, 5  # 1 CLS + 4 registers
BACKBONE = "facebook/dinov3-vits16-pretrain-lvd1689m"
IMAGES = Path("/kaggle/input/takhi-parts/images")
LABELS = Path("/kaggle/input/takhi-parts/labels.json")
UNLABELED = Path("/kaggle/input/takhi-unlabeled")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
processor = AutoImageProcessor.from_pretrained(BACKBONE, use_fast=True)
backbone = DINOv3ViTModel.from_pretrained(BACKBONE).eval().to(device)
for p in backbone.parameters():
    p.requires_grad = False
labels = json.loads(LABELS.read_text(encoding="utf-8"))
print("labeled photos:", sum(1 for f in IMAGES.glob("*") if f.name in labels))

In [ ]:
def fill_polygon(mask, points, class_id):
    canvas = Image.new("L", (PATCHES, PATCHES), 0)
    draw = ImageDraw.Draw(canvas)
    draw.polygon([(p["x"] * PATCHES, p["y"] * PATCHES) for p in points], fill=class_id)
    return torch.maximum(mask, torch.tensor(list(canvas.getdata()), dtype=torch.long).view(PATCHES, PATCHES))

class PartsDataset(torch.utils.data.Dataset):
    def __init__(self, images, transform):
        self.images, self.transform = images, transform
    def __len__(self):
        return len(self.images)
    def __getitem__(self, index):
        path = self.images[index]
        with Image.open(path) as image:
            image = ImageOps.exif_transpose(image).convert("RGB")
        pixel_values = self.transform(image, return_tensors="pt")["pixel_values"][0]
        mask = torch.zeros(PATCHES, PATCHES, dtype=torch.long)
        for polygon in labels.get(path.name, []):
            mask = fill_polygon(mask, polygon["points"], CLASSES.index(polygon["className"]))
        return pixel_values, mask

class PartsModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.head = nn.Linear(backbone.config.hidden_size, len(CLASSES))
    def forward(self, pixel_values):
        with torch.no_grad():
            hidden = backbone(pixel_values).last_hidden_state[:, OFFSET:]
        b, n, _ = hidden.shape
        grid = int(n ** 0.5)
        logits = self.head(hidden).transpose(1, 2).reshape(b, len(CLASSES), grid, grid)
        return F.interpolate(logits, size=(SIZE, SIZE), mode="bilinear", align_corners=False)

In [ ]:
transform = lambda image, **kw: processor(images=image, return_tensors="pt", **kw)
labeled = sorted(p for p in IMAGES.glob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"} and p.name in labels)
model = PartsModel().to(device)
optimizer = torch.optim.AdamW(model.head.parameters(), lr=3e-4, weight_decay=1e-3)

def train_one_epoch(loader):
    model.train()
    total = correct = 0
    for pixel_values, masks in loader:
        pixel_values, masks = pixel_values.to(device), masks.to(device)
        logits = model(pixel_values)
        logits = F.interpolate(logits, size=masks.shape[-2:], mode="bilinear", align_corners=False)
        loss = F.cross_entropy(logits, masks)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        total += masks.numel(); correct += (logits.argmax(1) == masks).sum().item()
    return correct / total

loader = torch.utils.data.DataLoader(PartsDataset(labeled, transform), batch_size=4, shuffle=True)
for epoch in range(25):
    acc = train_one_epoch(loader)
    if (epoch + 1) % 5 == 0:
        print(f"epoch {epoch + 1}: pixel accuracy {acc:.3f}", flush=True)

In [ ]:
# Self-training: уверенные пиксели неразмеченных фото становятся псевдоразметкой.
if UNLABELED.is_dir():
    pool = sorted(p for p in UNLABELED.glob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"})
    for round_index in range(3):
        added = 0
        model.eval()
        for path in pool:
            if path.name in labels:
                continue
            with Image.open(path) as image:
                image = ImageOps.exif_transpose(image).convert("RGB")
            inputs = transform(image, return_tensors="pt")
            with torch.no_grad():
                probs = F.softmax(model(inputs["pixel_values"].to(device))[0], dim=0).cpu()
            confident = probs.max(0).values > 0.9
            for class_id in (1, 2, 3):
                class_mask = confident & (probs.argmax(0) == class_id)
                if class_mask.float().mean() <= 0.005:
                    continue
                ys, xs = torch.where(class_mask)
                x1, x2 = xs.min().item() / PATCHES, xs.max().item() / PATCHES
                y1, y2 = ys.min().item() / PATCHES, ys.max().item() / PATCHES
                labels.setdefault(path.name, []).append({"className": CLASSES[class_id],
                    "points": [{"x": x1, "y": y1}, {"x": x2, "y": y1}, {"x": x2, "y": y2}, {"x": x1, "y": y2}]})
            added += 1
        labeled = sorted({p.name: p for p in list(IMAGES.glob("*")) + pool if p.name in labels}.values())
        print(f"round {round_index}: pseudo-labeled {added} photos", flush=True)
        loader = torch.utils.data.DataLoader(PartsDataset(labeled, transform), batch_size=4, shuffle=True)
        for epoch in range(5):
            acc = train_one_epoch(loader)
        print(f"round {round_index} retrain accuracy {acc:.3f}", flush=True)

In [ ]:
# Экспорт TorchScript для локального сервиса (ml/parts_segment.py).
example = torch.zeros(1, 3, SIZE, SIZE)
traced = torch.jit.trace(model.eval().cpu(), example)
traced.save("/kaggle/working/horse_parts.pt")
print("saved /kaggle/working/horse_parts.pt")

## Дальше

Скачайте `/kaggle/working/horse_parts.pt` и положите в проект: `data/ml/horse_parts.pt`. Перезапустите `npm start` — в интерфейсе появится кнопка «Части тела».